# 1. Environment

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

In [2]:
BASE_DIR = Path(r"C:\\Users\\yatsyk\\Documents\\NextCloud_Synchro\\passim_BoH")
os.chdir(BASE_DIR)
BASE_DIR

FileNotFoundError: [WinError 3] Le chemin d’accès spécifié est introuvable: 'C:\\Users\\yatsyk\\Documents\\NextCloud_Synchro\\passim_BoH'

# 2. Paths

In [5]:
reference_csv_path = Path("01.data/01.text_editions/texts_from_Heurist/ref_all_texts.csv")
csv_folder_path = Path("04.CoMMa/00.data/03.csvs_arkindex/EXACT_MATCH")

source_texts_dir = Path("02.passim/00.passim_input/source_texts")
manuscript_jsonl_dir = Path("02.passim/00.passim_input/comma_arkindex_mss/by_page")
all_texts_output_dir = Path("02.passim/00.passim_input/comma_arkindex_mss/all_texts")
long_texts_output_dir = Path("02.passim/00.passim_input/comma_arkindex_mss/texts_over_200_chars")

source_texts_dir.mkdir(parents=True, exist_ok=True)
manuscript_jsonl_dir.mkdir(parents=True, exist_ok=True)
all_texts_output_dir.mkdir(parents=True, exist_ok=True)
long_texts_output_dir.mkdir(parents=True, exist_ok=True)

reference_csv_path, csv_folder_path

(WindowsPath('01.data/01.text_editions/texts_from_Heurist/ref_all_texts.csv'),
 WindowsPath('04.CoMMa/00.data/03.csvs_arkindex/EXACT_MATCH'))

# 3. Helper functions

## 3.1. Build reference JSONL files

In [6]:
def build_reference_jsonl(reference_csv, all_output_path, min_chars=200):
    df = pd.read_csv(reference_csv)
    df = df.dropna(subset=["Text"])
    df["Text"] = df["Text"].astype(str).str.strip()
    df = df[df["Text"] != ""]

    all_records = df.apply(
        lambda row: {
            "id": row["Work H-ID"],
            "series": "source",
            "text": row["Text"],
        },
        axis=1,
    ).tolist()

    long_df = df[df["Text"].str.len() > min_chars].copy()
    long_records = long_df.apply(
        lambda row: {
            "id": row["Work H-ID"],
            "series": "source",
            "text": row["Text"],
        },
        axis=1,
    ).tolist()

    with open(all_output_path, "w", encoding="utf-8") as handle:
        for record in all_records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


    print(f"Saved {len(all_records)} reference texts to {all_output_path}")

    return df, long_df

## 3.2. Convert Arkindex CSV files into manuscript JSONL files

In [7]:
def csv_to_manuscript_jsonl(csv_path, output_path):
    df = pd.read_csv(csv_path)

    required_columns = {"page_ordering", "text"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns in {csv_path.name}: {sorted(missing_columns)}")

    df = df.sort_values("page_ordering").copy()
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    df = df[df["text"] != ""]

    manuscript_id = csv_path.stem
    records = [
        {
            "id": int(row.page_ordering),
            "series": manuscript_id,
            "text": row.text,
        }
        for row in df.itertuples(index=False)
    ]

    with open(output_path, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    return len(records)


def convert_csv_folder_to_jsonl(input_folder, output_folder):
    output_folder.mkdir(parents=True, exist_ok=True)
    summary = []

    for csv_path in sorted(input_folder.glob("*.csv")):
        output_path = output_folder / f"{csv_path.stem}_HTR.jsonl"
        row_count = csv_to_manuscript_jsonl(csv_path, output_path)
        summary.append(
            {
                "manuscript_id": csv_path.stem,
                "jsonl_path": str(output_path),
                "non_empty_pages": row_count,
            }
        )

    summary_df = pd.DataFrame(summary)
    print(f"Converted {len(summary_df)} CSV files into manuscript JSONL files")
    return summary_df

## 3.3. Combine reference texts and manuscript transcription into `in.json`

In [8]:
def load_and_combine_json(ref_file, ms_file):
    combined_data = []
    for filename in [ref_file, ms_file]:
        with open(filename, "r", encoding="utf-8") as file:
            data = [json.loads(line) for line in file]
            combined_data.extend(data)
    return combined_data


def create_folders_with_combined_json(ref_path, folder_path, output_folder):
    output_folder.mkdir(parents=True, exist_ok=True)
    ms_files = sorted(folder_path.glob("*.jsonl"))

    if not ms_files:
        print(f"No .jsonl files found in {folder_path}")
        return

    for ms_file_path in ms_files:
        docs = load_and_combine_json(ref_path, ms_file_path)
        base_name = ms_file_path.stem.replace("_HTR", "") + "_in_for_passim"
        output_subfolder = output_folder / base_name
        output_subfolder.mkdir(parents=True, exist_ok=True)

        in_json_path = output_subfolder / "in.json"
        with open(in_json_path, "w", encoding="utf-8") as handle:
            for doc in docs:
                handle.write(json.dumps(doc, ensure_ascii=False) + "\n")

        print(f"Created {in_json_path}")

# 4. Export reference texts for passim

In [9]:
ref_all_jsonl_path = source_texts_dir / "ref_all_texts.jsonl"

all_texts_df, long_texts_df = build_reference_jsonl(
    reference_csv=reference_csv_path,
    all_output_path=ref_all_jsonl_path,
)

all_texts_df

Saved 14814 reference texts to 02.passim\00.passim_input\source_texts\ref_all_texts.jsonl


,Work H-ID,rec_Title,Type,textLang,Title,Text,Liturgical function,Note,Textual source,Witness (note),Id Horae,ID Arkindex,text_length
88,395034,A cunctis malis iminentibus erripiat nos omnip...,Level 4,NaN,NaN,A cunctis malis iminentibus erripiat nos omnip...,Benediction,NaN,NaN,NaN,1.0,0,70.0
89,395035,Ab hoste maligno eripiat nos Dei genitrix virg...,Level 4,NaN,NaN,Ab hoste maligno eripiat nos Dei genitrix virgo,Benediction,NaN,NaN,NaN,10.0,2f32f250-b269-4eed-a8f9-019da83619cb,47.0
90,395036,Ave magnifica misericordiae | Cantus ID 200457...,Level 4,NaN,NaN,Ave magnifica misericordiae,Antiphon,NaN,NaN,NaN,100.0,0,27.0
91,395037,Que cum audisset turbata est in sermone | HORA...,Level 4,NaN,NaN,Que cum audisset turbata est in sermone,Lesson,NaN,Lk 1:29-33,NaN,1000.0,3ef3cccd-6464-42c6-94c9-720e227f5519,39.0
92,395038,Quem credimus ex virgine natum | HORAE | | Hym...,Level 4,NaN,NaN,Quem credimus ex virgine natum,Hymn,NaN,NaN,NaN,1001.0,0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15129,725238,Precibus et meritis beatissime semper virginis...,Level 4,NaN,NaN,Precibus et meritis beatissime semper virginis...,Absolution,NaN,NaN,NaN,14811.0,e450e12c-fc24-43d9-9b27-1e6e5b480bdd,109.0
15130,725239,Precibus et meritis beatissime semperque virgi...,Level 4,NaN,NaN,Precibus et meritis beatissime semperque virgi...,Absolution,NaN,NaN,NaN,14812.0,0b750036-2e5c-43e9-ae5a-142e5e9ef8ee,112.0
15131,725240,Precibus et meritis beatissime semperque virgi...,Level 4,NaN,NaN,Precibus et meritis beatissime semperque virgi...,Absolution,NaN,NaN,NaN,14813.0,8929cc0b-966a-4563-b764-df2d30f9ecfa,117.0
15132,725241,Per virginem Mariam concedat nobis Dominus sal...,Level 4,NaN,NaN,Per virginem Mariam concedat nobis Dominus sal...,Benediction,NaN,NaN,NaN,14814.0,87c212ac-ed76-4bce-9eca-b16f66af0b09,59.0


# 5. Convert CoMMa Arkindex CSV files into manuscript JSONL files

In [10]:
conversion_summary = convert_csv_folder_to_jsonl(
    input_folder=csv_folder_path,
    output_folder=manuscript_jsonl_dir,
)

conversion_summary

Converted 178 CSV files into manuscript JSONL files


,manuscript_id,jsonl_path,non_empty_pages
0,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,02.passim\00.passim_input\comma_arkindex_mss\b...,125
1,0068e751-c249-4d8b-82f6-07933fe0cecd,02.passim\00.passim_input\comma_arkindex_mss\b...,135
2,009c37f0-618a-4d58-b81a-511a36fa0fac,02.passim\00.passim_input\comma_arkindex_mss\b...,288
3,02181622-2bcc-4b35-b124-63c701e25860,02.passim\00.passim_input\comma_arkindex_mss\b...,234
4,0253a5a1-0667-4c1c-b499-a59d48abc884,02.passim\00.passim_input\comma_arkindex_mss\b...,280
...,...,...,...
173,f411cb6b-7e69-4e08-bdc8-282b62469daa,02.passim\00.passim_input\comma_arkindex_mss\b...,449
174,f8ac87c1-b602-4093-8bde-419bf7a38c3d,02.passim\00.passim_input\comma_arkindex_mss\b...,267
175,f8ef053d-ce9e-4399-a79e-a93bc92457d0,02.passim\00.passim_input\comma_arkindex_mss\b...,215
176,f9f1fca3-072f-4056-9144-8aae1d4a6504,02.passim\00.passim_input\comma_arkindex_mss\b...,205


# 6. Create `in.json` files for passim

## 6.1. With all reference texts

In [11]:
create_folders_with_combined_json(
    ref_path=ref_all_jsonl_path,
    folder_path=manuscript_jsonl_dir,
    output_folder=all_texts_output_dir,
)

Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\003d5be0-3453-46e9-b904-4f6d4f1ee4b5_in_for_passim\in.json
Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\0068e751-c249-4d8b-82f6-07933fe0cecd_in_for_passim\in.json
Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\009c37f0-618a-4d58-b81a-511a36fa0fac_in_for_passim\in.json
Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\02181622-2bcc-4b35-b124-63c701e25860_in_for_passim\in.json
Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\0253a5a1-0667-4c1c-b499-a59d48abc884_in_for_passim\in.json
Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\031dfdd8-d3a9-4718-bcc0-32b946a705ac_in_for_passim\in.json
Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\034c35ae-17e3-4e51-9821-8cf1662b1253_in_for_passim\in.json
Created 02.passim\00.passim_input\comma_arkindex_mss\all_texts\05906f2e-0c51-472d-a1f6-b8d59502057b_in_for_passim\in.json
Created 02.passim\00.pas